<a href="https://colab.research.google.com/github/grasht/grashaw_GAN_research_project/blob/main/GANS_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#%pip install torch matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

#Pre-Processing


*   Handle Missing Values - Create a binary column that tracks if the value was missing, then mean imputation for continuous values and mode imputation for categorical values.
*   One Hot Encoding for categorical columns that are small. Embedding based for large categories. Conditional Vectors may be consider for special cases.
*   Normalization - Min-Max or Standardization
*   Log Transform Highly Skewed Data
*   Class Balancing for rare categories (oversample rare values)

# Post Processing
*   Clip Features and Enforce Bounds
*   List item





#CTGAN on Vehicle Sales

In [ ]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.8 MB/s eta 0:00:00


In [ ]:
!pip install table_evaluator

#Import The Data

In [ ]:
#!rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

base_path = "/content/drive/MyDrive/datasets/vehicle_sales"

!ls

In [ ]:
import pandas as pd

vs_df = pd.read_csv(base_path+"/train.csv")  # filename may vary slightly
print(vs_df.head())
print(vs_df.shape)

In [ ]:
import pandas as pd

def handle_missing_continuous(df_in, continuous_cols, strategy="median"):
    df = df_in.copy()

    for col in continuous_cols:
        # 1. Create missing indicator
        df[f"{col}_missing"] = df[col].isna().astype(int)

        # 2. Impute values
        if strategy == "median":
            fill_value = df[col].median()
        elif strategy == "mean":
            fill_value = df[col].mean()
        elif strategy == "zero":
            fill_value = 0
        else:
            raise ValueError("Unsupported strategy")

        df[col] = df[col].fillna(fill_value)

    return df

In [ ]:
vs_df = handle_missing_continuous(
    vs_df,
    continuous_cols=["year", "condition", "odometer", "mmr", "sellingprice"]
)

vs_df = vs_df.drop(columns=["vin"])



# Step 1: clean string
vs_df["saledate"] = vs_df["saledate"].astype(str)
vs_df["saledate"] = vs_df["saledate"].str.replace(r"GMT.*", "", regex=True)

# Step 2: parse safely
vs_df["saledate"] = pd.to_datetime(vs_df["saledate"], errors="coerce")

# Step 3: inspect failures
print(vs_df["saledate"].isna().sum())

# Step 4: handle missing
vs_df = vs_df[vs_df["saledate"].notna()]

vs_df["saledate"] = pd.to_datetime(vs_df["saledate"], errors="coerce")

vs_df["saledate"] = vs_df["saledate"].dt.tz_localize(None)
vs_df["sale_year"] = vs_df["saledate"].dt.year
vs_df["sale_month"] = vs_df["saledate"].dt.month
vs_df["sale_day"] = vs_df["saledate"].dt.day
vs_df["sale_hour"] = vs_df["saledate"].dt.hour
vs_df = vs_df.drop(columns=["saledate"])

print(vs_df.head())

In [ ]:
vs_df.isnull().values.any()

vs_df.isnull().sum()

In [ ]:
from ctgan import CTGAN

ctgan = CTGAN()
ctgan.fit(vs_df, discrete_columns = [
    "make",
    "model",
    "trim",
    "body",
    "transmission",
    "state",
    "color",
    "interior",
    "seller"
])


samples = ctgan.sample(1000)